# 06 — Оценка параметров модели по дневным данным S&P 500

Практическая часть, шаг 1. По рецепту из `docs/06-empirical.md`:

1. время измеряется в месяцах по `D=21` торговых дней, один день — `1/D`;
2. в окне из `W` дневных лог-приростов оцениваются снос `a = D·mean(r)`, показатель
   Херста `H` R/S-анализом приростов лог-цены без тренда, масштаб `σ = СКО(r)·D^H`
   и нормированный снос `μ = a/σ`;
3. доверительные интервалы оценок — параметрический бутстреп;
4. скользящие оценки накладываются на практические границы `μ₁(H)`, `μ₂(H)`.

Ноутбук не обучает стратегий. Границы читаются из дневного прогона
(`03-grid.ipynb`, `RUN_ID="daily21_train20k_test20k_v1"`); пока его нет,
используется теоретический прогон с предупреждением.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
C1, C2, C3, C4, C5 = "#2a78d6", "#eb6834", "#1baf7a", "#8e63d2", "#b8860b"
INK, MUTED = "#0b0b0b", "#8a8a86"
plt.rcParams.update({
    "figure.figsize": (9, 4.5), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.grid": True, "grid.color": "#e8e8e4", "grid.linewidth": 0.8,
    "lines.linewidth": 1.6, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9,
    "legend.frameon": False,
})

from osfbm.data import block_frame, fetch_prices, month_blocks
from osfbm.estimation import (bootstrap_estimates, estimate, percentile_ci,
                              rolling_estimates)
from osfbm.policy_bank import CASH, HOLD, POLICY, Boundaries


## Параметры

`WINDOW` — базовое окно оценки в торговых днях; `WINDOWS` — окна для сравнения.
`N_BOOT` — число бутстреп-реплик на одну дату; `BOOT_EVERY=1` считает интервалы
для каждого месяца (несколько минут), большее значение прореживает даты.

In [ ]:
TICKER = "^GSPC"
REFRESH_DATA = False          # True: заново скачать котировки с Yahoo Finance
D = 21
WINDOW = 504
WINDOWS = (126, 252, 504, 1008)
HURST_METHOD = "rs"           # "rs" (R/S, основной) или "dfa" (контроль)
N_BOOT = 200
BOOT_EVERY = 1
CONFIDENCE = 0.95
EPSILON = 0.04
DAILY_RUN_ID = "daily21_train20k_test20k_v1"
THEORY_RUN_ID = "fresh_train20k_test20k_v1"
RUNS = ROOT / "result_optimal_stopping" / "runs"
OUT = ROOT / "result_optimal_stopping" / "empirical" / "estimation"
OUT.mkdir(parents=True, exist_ok=True)

## Данные

Дневные закрытия с Yahoo Finance кэшируются в `data/gspc_daily.csv`. Первый запуск
требует сети и установленного `yfinance` (`uv sync`).

In [ ]:
try:
    prices = fetch_prices(TICKER, refresh=REFRESH_DATA)
except ImportError as error:
    raise SystemExit("Установите зависимости: uv sync (нужен yfinance)") from error
logp = np.log(prices.to_numpy())
dates = prices.index
print(f"{TICKER}: {len(prices)} закрытий, {dates[0].date()} — {dates[-1].date()}")

fig, ax = plt.subplots()
ax.plot(dates, logp, color=C1, linewidth=1)
ax.set(title=f"ln S_t, {TICKER}", xlabel="дата", ylabel="ln S")
plt.show()

## Оценки по всей истории и по десятилетиям

Контрольная таблица: как параметры выглядят на длинных отрезках. Оба метода оценки
`H` показаны рядом; R/S с поправкой Анниса–Ллойда систематически сжимает `H` к 0.5
(см. синтетическую проверку в 07).

In [ ]:
def estimate_row(segment, label):
    rs, dfa = estimate(segment, D, "rs"), estimate(segment, D, "dfa")
    return {"период": label, "дней": len(segment) - 1, "a (в месяц)": rs.a,
            "σ (в месяц)": rs.sigma, "H (R/S)": rs.H_raw, "H (DFA)": dfa.H_raw,
            "μ = a/σ": rs.mu, "σ при H=DFA": dfa.sigma, "μ при H=DFA": dfa.mu}

rows = [estimate_row(logp, "вся история")]
years = dates.year
for decade in range(years.min() // 10 * 10, years.max() + 1, 10):
    mask = (years >= decade) & (years < decade + 10)
    if mask.sum() > 300:
        rows.append(estimate_row(logp[mask], f"{decade}-е"))
long_run = pd.DataFrame(rows)
display(long_run.style.format(precision=4))
long_run.to_csv(OUT / "long_run_estimates.csv", index=False)

## Скользящие оценки

Для каждого месяца (блока из `D` дней) параметры оцениваются по `W` приростам,
заканчивающимся в точке решения `t₀`. Будущее блока не используется.

In [ ]:
blocks = month_blocks(len(logp), D, warmup=max(WINDOWS))
frame = block_frame(dates, blocks)
rolling = {}
for w in WINDOWS:
    est = rolling_estimates(logp, blocks[:, 0], w, D, HURST_METHOD)
    rolling[w] = pd.concat([frame, est.drop(columns="decision_idx")], axis=1)
    rolling[w].to_csv(OUT / f"rolling_W{w}_{HURST_METHOD}.csv", index=False)
print("Месяцев:", len(frame), "первое решение:", frame.decision_date.iloc[0].date())

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for w, color in zip(WINDOWS, (C4, C3, C1, C2)):
    axes[0].plot(rolling[w].decision_date, rolling[w].H, color=color, label=f"W={w}",
                 linewidth=1.2 if w == WINDOW else 0.8, alpha=1 if w == WINDOW else 0.7)
    axes[1].plot(rolling[w].decision_date, rolling[w].mu, color=color, label=f"W={w}",
                 linewidth=1.2 if w == WINDOW else 0.8, alpha=1 if w == WINDOW else 0.7)
axes[0].axhline(0.5, color=MUTED, linewidth=0.8)
axes[0].set(ylabel="Ĥ", title=f"Скользящие оценки ({HURST_METHOD.upper()})")
axes[1].axhline(0, color=MUTED, linewidth=0.8)
axes[1].set(ylabel="μ̂ = â/σ̂", xlabel="дата решения")
axes[0].legend(ncol=len(WINDOWS))
fig.tight_layout(); fig.savefig(OUT / "rolling_H_mu.png", dpi=160, bbox_inches="tight")
plt.show()

base = rolling[WINDOW]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
axes[0].plot(base.decision_date, base.a, color=C1); axes[0].axhline(0, color=MUTED, linewidth=0.8)
axes[0].set(title=f"â в месяц, W={WINDOW}", xlabel="дата")
axes[1].plot(base.decision_date, base.sigma, color=C2)
axes[1].set(title=f"σ̂ в месяц, W={WINDOW}", xlabel="дата")
fig.tight_layout(); fig.savefig(OUT / f"rolling_a_sigma_W{WINDOW}.png", dpi=160, bbox_inches="tight")
plt.show()
display(base[["a", "sigma", "H", "mu"]].describe().T.style.format(precision=4))

## Доверительные интервалы параметров

Параметрический бутстреп: по оценённым `(â, σ̂, Ĥ)` симулируются `N_BOOT` окон той
же длины, параметры переоцениваются, берутся перцентили. Интервал описывает
статистическую неопределённость оценки при верной модели; смещение метода он не
устраняет.

In [ ]:
selected = base.iloc[::BOOT_EVERY]
ci_rows = []
for _, row in selected.iterrows():
    est = estimate(logp[int(row.decision_idx) - WINDOW: int(row.decision_idx) + 1], D, HURST_METHOD)
    boot = bootstrap_estimates(est, WINDOW, n_boot=N_BOOT, seed=int(row.decision_idx))
    record = {"decision_date": row.decision_date, "H": est.H, "mu": est.mu, "sigma": est.sigma, "a": est.a}
    for name in ("H", "mu", "sigma", "a"):
        record[f"{name}_low"], record[f"{name}_high"] = percentile_ci(boot[name], CONFIDENCE)
    ci_rows.append(record)
ci = pd.DataFrame(ci_rows)
ci.to_csv(OUT / f"bootstrap_ci_W{WINDOW}_{HURST_METHOD}.csv", index=False)

fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
for ax, name, color, ref in ((axes[0], "H", C1, 0.5), (axes[1], "mu", C2, 0.0)):
    ax.fill_between(ci.decision_date, ci[f"{name}_low"], ci[f"{name}_high"], color=color, alpha=0.2,
                    label=f"бутстреп {100*CONFIDENCE:g}%")
    ax.plot(ci.decision_date, ci[name], color=color, label="оценка")
    ax.axhline(ref, color=MUTED, linewidth=0.8)
    ax.set(ylabel=name)
    ax.legend()
axes[0].set(title=f"Оценки с интервалами, W={WINDOW}, {HURST_METHOD.upper()}")
fig.tight_layout(); fig.savefig(OUT / f"bootstrap_ci_W{WINDOW}.png", dpi=160, bbox_inches="tight")
plt.show()
last = ci.iloc[-1]
print(f"Последняя дата {last.decision_date.date()}: "
      f"H={last.H:.3f} [{last.H_low:.3f}, {last.H_high:.3f}], "
      f"μ={last.mu:.3f} [{last.mu_low:.3f}, {last.mu_high:.3f}]")

## Оценки на фоне практических границ

Ключевая картинка главы: где S&P 500 живёт в плоскости `(H, μ)` относительно зон
«не покупать» (`μ ≤ μ₁(H)`), «держать до конца» (`μ ≥ μ₂(H)`) и зоны, где нужна
стратегия остановки. Границы дневного прогона и теоретического прогона рисуются
вместе, если оба доступны.

In [ ]:
def load_boundaries(run_id, label):
    run_dir = RUNS / run_id
    csv = run_dir / "analysis" / f"epsilon_{EPSILON:g}" / f"confidence_{CONFIDENCE:g}" / "boundaries.csv"
    if csv.exists():
        return Boundaries.from_csv(csv, EPSILON, label)
    if (run_dir / "experiment.json").exists():
        return Boundaries.from_run(run_dir, EPSILON, label)
    return None

daily = load_boundaries(DAILY_RUN_ID, "дневная сетка N=21")
theory = load_boundaries(THEORY_RUN_ID, "теоретическая сетка N=20, n_fine=200")
if daily is None:
    print(f"Дневного прогона {DAILY_RUN_ID} ещё нет: выполните 03-grid.ipynb. "
          "Режимы ниже считаются по теоретическому прогону.")
active = daily or theory
if active is None:
    raise SystemExit("Нет ни одного прогона с границами")
print("Границы для режимов:", active.label)

fig, ax = plt.subplots(figsize=(9, 6))
for b, ls in ((daily, "-"), (theory, "--")):
    if b is None:
        continue
    f = b.frame()
    ax.plot(f.H, f.mu1, ls, color=C1, label=f"μ₁(H), {b.label}")
    ax.plot(f.H, f.mu2, ls, color=C2, label=f"μ₂(H), {b.label}")
decade = (base.decision_date.dt.year // 10 * 10)
scatter = ax.scatter(base.H, base.mu, c=decade, cmap="viridis", s=10, alpha=0.7)
fig.colorbar(scatter, ax=ax, label="десятилетие")
ax.axhline(0, color=MUTED, linewidth=0.8)
ax.set(xlabel="Ĥ", ylabel="μ̂", title=f"S&P 500, скользящие оценки W={WINDOW}, ε={EPSILON:g}",
       ylim=(np.nanmin(base.mu) - 0.5, np.nanmax(base.mu) + 0.5))
ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig(OUT / f"plane_H_mu_W{WINDOW}.png", dpi=160, bbox_inches="tight")
plt.show()

shares = {}
for w in WINDOWS:
    reg = pd.Series([active.regime(h, m) for h, m in zip(rolling[w].H, rolling[w].mu)])
    shares[w] = reg.value_counts(normalize=True).reindex([CASH, POLICY, HOLD], fill_value=0)
shares = pd.DataFrame(shares).T.rename_axis("W")
print("Доля месяцев по режимам:")
display(shares.style.format("{:.1%}"))
shares.to_csv(OUT / "regime_shares.csv")

if "ci" in globals():
    last = ci.iloc[-1]
    est = estimate(logp[-WINDOW - 1:], D, HURST_METHOD)
    boot = bootstrap_estimates(est, WINDOW, n_boot=N_BOOT, seed=1)
    probs = pd.Series([active.regime(h, m) for h, m in zip(boot.H, boot.mu)]).value_counts(normalize=True)
    print(f"Последнее окно: режим {active.regime(est.H, est.mu)}; бутстреп-вероятности режимов:")
    display(probs.reindex([CASH, POLICY, HOLD], fill_value=0).to_frame("P").style.format("{:.1%}"))

## Сравнение границ: дневная сетка против теоретической

Разница между границами при `n_fine=200` и `n_fine=21` — цена того, что стратегия
видит путь только раз в день. Таблица появится, когда готовы оба прогона.

In [ ]:
if daily is not None and theory is not None:
    merged = daily.frame().merge(theory.frame(), on="H", suffixes=("_daily", "_theory"))
    merged["Δμ1"] = merged.mu1_daily - merged.mu1_theory
    merged["Δμ2"] = merged.mu2_daily - merged.mu2_theory
    display(merged.style.format(precision=3, na_rep="—"))
    merged.to_csv(OUT / "boundaries_daily_vs_theory.csv", index=False)
else:
    print("Нужны оба прогона для сравнения границ.")
print("Результаты:", OUT)